# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

2次元の四分木の構築を学ぶ教材です。粒子は円盤内に一様配置し、力には3次元Newton重力の逆二乗則を用います。同一位置・領域外・最大深度超過の入力はエラーとして扱います。動画保存にはPillowを使用します。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.animation import FuncAnimation
from collections import defaultdict
parameters = {
    'axes.labelsize': 26,      # Set the font size for x and y axis labels to 26
    'xtick.labelsize': 24,     # Set the font size for x-axis ticks to 24
    'ytick.labelsize': 24,     # Set the font size for y-axis ticks to 24
    'axes.titlesize': 24,      # Set the font size for axis titles to 24
    'figure.titlesize': 24,    # Set the font size for the overall figure title to 24
    'legend.fontsize': 22,     # Set the font size for the legend to 22
    'figure.figsize': [12, 8], # Set the default figure size to 12 inches wide and 8 inches high
    'axes.linewidth': 2.0      # Set the line width for the borders of the axes to 2.0
}
plt.rcParams.update(parameters)
np.random.seed(20241108)  # Reproducible teaching examples


In [ ]:
# Define colors for different depths
depth_colors = ['red', 'orange', 'green', 'blue', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']

## Code Explanation

### Objective
This code implements a **QuadTree** structure for efficiently organizing particles (bodies) with position and mass in 2D space. It is commonly used in simulations like the **Barnes-Hut algorithm** to approximate interactions such as gravitational forces between multiple bodies.

### Key Components

#### 1. `Body` Class
- **Purpose**: Represents a particle with a 2D position and mass.
- **Attributes**:
  - `position`: 2D coordinates of the body (in numpy array format).
  - `mass`: The mass of the body.

#### 2. `QuadNode` Class
- **Purpose**: Represents a node in the QuadTree, which divides space into four quadrants (SW, NW, SE, NE) to manage spatial data.
- **Attributes**:
  - `boundary`: Spatial boundary of the node as a tuple (`x_min`, `x_max`, `y_min`, `y_max`).
  - `mass`: Total mass contained in the node.
  - `center_of_mass`: The center of mass of all bodies within the node.
  - `children`: List of four child nodes (SW, NW, SE, NE), initially `None`.
  - `is_divided`: A flag indicating whether the node has been subdivided.
  - `max_depth`: The maximum allowable depth of the QuadTree to avoid infinite subdivisions.

### Key Methods

- **`insert(body)`**:
  - **Purpose**: Adds a body to the QuadTree.
  - If the node is empty, the body is stored directly. If the node already contains a body and has not been subdivided, the node is split into four child nodes. The center of mass and total mass of the node are updated after insertion.

- **`subdivide()`**:
  - **Purpose**: Divides the current node into four child nodes (SW, NW, SE, NE). This happens when a second body needs to be inserted into a node that already contains a body.

- **`update_center_of_mass(body)`**:
  - **Purpose**: Updates the node’s center of mass and total mass by incorporating the mass and position of a new body.

- **`get_child_index(position)`**:
  - **Purpose**: Determines the appropriate child node (SW, NW, SE, NE) based on the position of a body.

- **`contains(position)`**:
  - **Purpose**: Checks whether a body’s position lies within the node’s boundaries.

In [ ]:
# Body class representing a particle with position and mass
class Body:
    def __init__(self, position, mass):
        self.position = np.array(position)
        self.mass = mass

# QuadNode class representing each node in the QuadTree
class QuadNode:
    _id_counter = 0  # Class variable to assign unique IDs to nodes

    @classmethod
    def reset_id_counter(cls):
        cls._id_counter = 0
        
    def __init__(self, boundary, parent=None, max_depth=10):
        """
        Initializes a QuadNode.
        
        Parameters:
        - boundary: Tuple representing the node's boundaries (x_min, x_max, y_min, y_max)
        - parent: Reference to the parent QuadNode
        - max_depth: Maximum allowed depth to prevent infinite subdivisions
        """
        self.id = QuadNode._id_counter
        QuadNode._id_counter += 1
        print(f"QuadNode created with ID: {self.id}")  # Debugging line

        self.boundary = boundary  # Node boundary (x_min, x_max, y_min, y_max)
        self.mass = 0
        self.center_of_mass = np.zeros(2)
        self.body = None
        self.children = [None, None, None, None]  # SW, NW, SE, NE
        self.is_divided = False  # Indicates whether the node has been subdivided
        self.parent = parent  # Reference to the parent node
        self.depth = parent.depth + 1 if parent else 0  # Current depth in the tree
        self.max_depth = max_depth  # Maximum allowed depth

    def insert(self, body):
        """
        Inserts a Body into the QuadTree.
        
        Parameters:
        - body: Instance of Body to be inserted
        
        Returns:
        - True if insertion was successful, False otherwise
        """
        if not self.contains(body.position):
            return False

        # If the node is empty and not divided, place the body here
        if self.body is None and not self.is_divided:
            self.body = body
            self.mass += body.mass
            self.center_of_mass = body.position
            print(f"Inserting body into node {self.id}")  # Debugging line
            return True

        # If the node already contains a body and is not divided, subdivide
        if self.body is not None and not self.is_divided:
            if self.depth >= self.max_depth:
                # Prevent infinite subdivision by not inserting additional bodies
                print(f"Maximum depth reached at node {self.id}. Cannot insert body at position {body.position}.")
                return False
            self.subdivide()
            # Re-insert the existing body into the appropriate child node
            index = self.get_child_index(self.body.position)
            inserted = self.children[index].insert(self.body)
            if not inserted:
                print(f"Failed to re-insert body at position {self.body.position} into child {index} of node {self.id}.")
            self.body = None  # Current node no longer holds the body
            self.is_divided = True

        # Insert the new body into the appropriate child node
        index = self.get_child_index(body.position)
        inserted = self.children[index].insert(body)
        if not inserted:
            print(f"Failed to insert body at position {body.position} into child {index} of node {self.id}.")
            return False

        # Update the center of mass and mass
        self.update_center_of_mass(body)
        return True

    def get_child_index(self, position):
        """
        Determines which child node the position belongs to.
        
        Returns:
        - Index (0: SW, 1: NW, 2: SE, 3: NE)
        """
        x_min, x_max, y_min, y_max = self.boundary
        mid_x = (x_min + x_max) / 2
        mid_y = (y_min + y_max) / 2

        if position[0] < mid_x:
            if position[1] < mid_y:
                return 0  # SW
            else:
                return 1  # NW
        else:
            if position[1] < mid_y:
                return 2  # SE
            else:
                return 3  # NE

    def subdivide(self):
        """
        Subdivides the current node into four child nodes: SW, NW, SE, NE.
        """
        x_min, x_max, y_min, y_max = self.boundary
        mid_x = (x_min + x_max) / 2
        mid_y = (y_min + y_max) / 2

        # Define child node boundaries in the order SW, NW, SE, NE
        self.children[0] = QuadNode((x_min, mid_x, y_min, mid_y), parent=self, max_depth=self.max_depth)  # SW
        self.children[1] = QuadNode((x_min, mid_x, mid_y, y_max), parent=self, max_depth=self.max_depth)  # NW
        self.children[2] = QuadNode((mid_x, x_max, y_min, mid_y), parent=self, max_depth=self.max_depth)  # SE
        self.children[3] = QuadNode((mid_x, x_max, mid_y, y_max), parent=self, max_depth=self.max_depth)  # NE

    def contains(self, position):
        """
        Checks if the position is within the node's boundary.
        
        Returns:
        - True if contained, False otherwise
        """
        x_min, x_max, y_min, y_max = self.boundary
        return x_min <= position[0] <= x_max and y_min <= position[1] <= y_max

    def update_center_of_mass(self, body):
        """
        Updates the node's center of mass and total mass.
        """
        total_mass = self.mass + body.mass
        if self.mass == 0:
            self.center_of_mass = body.position
        else:
            self.center_of_mass = (self.center_of_mass * self.mass + body.position * body.mass) / total_mass
        self.mass = total_mass

## Code Explanation

### Objective
This class defines the **Barnes-Hut algorithm**, which uses a **QuadTree** structure to efficiently compute gravitational forces between multiple bodies by approximating distant clusters of particles as a single mass. The key advantage of this method is reducing computational complexity for large-scale simulations.

### Key Components

#### 1. `BarnesHut` Class
- **Purpose**: Manages the construction of a **QuadTree** and calculates gravitational forces using the Barnes-Hut approximation.
- **Attributes**:
  - `root`: The root node of the QuadTree, representing the entire spatial region.
  - `max_depth`: The maximum allowed depth for the QuadTree to prevent infinite subdivision.

### Key Methods

- **`__init__(boundary, max_depth)`**:
  - Initializes the **BarnesHut** structure by setting up the root of the QuadTree.
  - `boundary`: Defines the spatial extent of the root node (x_min, x_max, y_min, y_max).
  - `max_depth`: Controls how deep the tree can be subdivided.

- **`insert(body)`**:
  - Inserts a `Body` (which has mass and position) into the QuadTree, organizing the bodies into different nodes for efficient computation.

- **`calculate_gravitational_force(body, theta=0.5)`**:
  - Computes the gravitational force acting on a given body by traversing the QuadTree and using the **Barnes-Hut** approximation for clusters of distant bodies.
  - `theta`: A threshold parameter that controls the level of approximation. A higher value means more approximation and faster computations, but less accuracy.

- **`_calculate_force(node, body, theta)`**:
  - A helper function that recursively calculates the gravitational force. If a node is far enough (based on `theta`), it approximates the force using the entire mass of the node's cluster.
  
- **`calculate_force_between(body, node)`**:
  - Computes the exact gravitational force between a body and another node in the QuadTree by calculating the force based on distance and mass.

- **`visualize_spatial_tree(node, ax)`**:
  - Visualizes the spatial division of the tree using **matplotlib**, showing how space is partitioned into quadrants (SW, NW, SE, NE).
  - Draws boundaries and highlights nodes that contain bodies.

- **`get_tree_edges_with_particles()`**:
  - Retrieves the edges between nodes that contain particles and their ancestors, useful for visualizing the tree structure.
  
- **`visualize_tree_structure()`**:
  - Uses **networkx** to visualize the hierarchical structure of the QuadTree, showing parent-child relationships between nodes.

- **`_calculate_depths_with_particles(node, depth=0, depths=None)`**:
  - Recursively calculates the depth of nodes in the QuadTree that contain particles. This helps to determine how deep the tree has subdivided and is used in visualization.

- **`_assign_positions_hierarchical()`**:
  - Assigns hierarchical positions to nodes for visualization purposes, ensuring that child nodes are placed below their parent nodes.

In [ ]:
# BarnesHut class managing the QuadTree
class BarnesHut:
    def __init__(self, boundary, max_depth=10):
        """
        Initializes the BarnesHut QuadTree.
        
        Parameters:
        - boundary: Tuple representing the root node's boundaries (x_min, x_max, y_min, y_max)
        - max_depth: Maximum allowed depth for the QuadTree
        """
        QuadNode._id_counter = 0  # クラス変数をリセット
        self._positions = set()
        self.root = QuadNode(boundary, max_depth=max_depth)

    def insert(self, body):
        """
        Inserts a Body into the QuadTree.
        """
        if not self.root.contains(body.position):
            raise ValueError("Particle outside root boundary")
        position = tuple(body.position)
        if position in self._positions:
            raise ValueError("Coincident particles are unsupported without softening")
        if not self.root.insert(body):
            raise ValueError("Tree insertion failed (maximum depth)")
        self._positions.add(position)

    def calculate_gravitational_force(self, body, theta=0.5):
        """
        Calculates the gravitational force on a body using the Barnes-Hut algorithm.
        
        Parameters:
        - body: Instance of Body for which to calculate the force
        - theta: Threshold parameter for approximation
        
        Returns:
        - Numpy array representing the gravitational force vector
        """
        return self._calculate_force(self.root, body, theta)

    def _calculate_force(self, node, body, theta):
        """
        Recursive helper function to calculate gravitational force.
        """
        if node.mass == 0 or (node.body is body and not node.is_divided):
            return np.zeros(2)  # No force from self

        distance = np.linalg.norm(body.position - node.center_of_mass)

        # Calculate the size of the node
        x_min, x_max, y_min, y_max = node.boundary
        node_size = max(x_max - x_min, y_max - y_min)

        if not node.is_divided or (not node.contains(body.position) and distance > 0 and node_size / distance < theta):
            # Use this node as an approximation
            force = self.calculate_force_between(body, node)
            return force
        else:
            # Subdivide and calculate force from child nodes
            force = np.zeros(2)
            for child in node.children:
                if child is not None:
                    force += self._calculate_force(child, body, theta)
            return force

    def calculate_force_between(self, body, node):
        """
        Calculates the gravitational force between a body and a node.
        """
        r = node.center_of_mass - body.position
        distance = np.linalg.norm(r)
        if distance == 0:
            return np.zeros(2)  # Avoid division by zero
        force_magnitude = G * body.mass * node.mass / distance**3
        force = force_magnitude * r
        return force

    def visualize_spatial_tree(self, node, ax):
        """
        Visualizes the spatial tree division.
        Only displays node IDs for nodes that contain particles.
        """
        # Draw the node boundary
        x_min, x_max, y_min, y_max = node.boundary
        edge_color = 'black'
        linewidth = 1
        rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=None, edgecolor=edge_color, linewidth=linewidth)
        ax.add_patch(rect)

        # Draw node ID only if the node contains a particle
        if node.body is not None:
            cx = (x_min + x_max) / 2
            cy = (y_min + y_max) / 2
            ax.text(cx, cy, str(node.id), color='green', fontsize=20, ha='center', va='center')

        # Draw the particle if it exists
        if node.body is not None:
            ax.scatter(node.body.position[0], node.body.position[1], color='blue', s=100, zorder=5)

        # Recursively draw child nodes if subdivided
        if node.is_divided:
            for child in node.children:
                if child is not None:
                    self.visualize_spatial_tree(child, ax)

    def get_tree_edges_with_particles(self):
        """
        Retrieves edges between nodes that contain particles and their ancestors.
        """
        edges = []

        def traverse(node):
            has_particle = False
            if node.body is not None:
                has_particle = True
            if node.is_divided:
                for child in node.children:
                    if child is not None:
                        child_has_particle = traverse(child)
                        if child_has_particle:
                            edges.append((node.id, child.id))
                            has_particle = True
            return has_particle

        traverse(self.root)
        return edges

    def visualize_tree_structure(self, font_size=16, alpha=0.7):
        """
        Visualizes the hierarchical tree structure using networkx.
        Nodes that do not contain particles are not displayed.
        
        Parameters:
        - font_size: Font size for node labels (default is 16)
        - alpha: Transparency level for nodes and edges (default is 0.7 for semi-transparent)
        """
        edges = self.get_tree_edges_with_particles()
        G = nx.DiGraph()
        G.add_edges_from(edges)

        # Calculate node depths
        node_depths = self._calculate_depths_with_particles(self.root)

        # Assign positions to nodes (child nodes are placed below parent nodes)
        pos = self._assign_positions_hierarchical(node_depths)

        # Define colors for different depths
        max_depth = max(node_depths.values()) if node_depths else 0
        color_map = depth_colors[:max_depth + 1]  # Slice the color list to the required number of depths

        # Assign colors based on node depth
        node_colors = [depth_colors[node_depths[node] % len(depth_colors)] for node in G.nodes()]

        # Plot the tree with labels, custom font size, and transparency (alpha)
        plt.figure(figsize=(12, 8))
        nx.draw(G, pos, with_labels=True, node_size=1000,
                node_color=node_colors, arrows=True,
                font_size=font_size,  # Set the font size here
                alpha=alpha,           # Set transparency level here
                arrowstyle='-|>', arrowsize=12)
        plt.title("QuadTree Hierarchical Structure")
        plt.show()

    def _calculate_depths_with_particles(self, node, depth=0, depths=None):
        """
        Calculates the depth of nodes that contain particles.
        """
        if depths is None:
            depths = {}
        if node.body is not None or node.is_divided:
            depths[node.id] = depth
            if node.is_divided:
                for child in node.children:
                    if child is not None and (child.body is not None or child.is_divided):
                        self._calculate_depths_with_particles(child, depth + 1, depths)
        return depths

    def _assign_positions_hierarchical(self, node_depths):
        """
        Assigns positions to each node hierarchically.
        Child nodes are placed below their parent nodes and evenly distributed horizontally.
        """
        pos = {}
        spacing_x = 1.0  # Horizontal spacing
        spacing_y = 1.5  # Vertical spacing

        # Group nodes by depth
        depth_to_nodes = {}
        for node_id, depth in node_depths.items():
            depth_to_nodes.setdefault(depth, []).append(node_id)

        # Counter for assigning horizontal positions
        x_counter = [0]

        def assign_positions(node, depth, pos, x_counter):
            if node.id not in node_depths:
                return
            if node.is_divided:
                for child in node.children:
                    if child is not None and child.id in node_depths:
                        assign_positions(child, depth + 1, pos, x_counter)
                # Set the parent node's x position as the mean of its children's x positions
                child_x = [pos[child.id][0] for child in node.children
                          if child is not None and child.id in node_depths]
                if child_x:
                    pos[node.id] = (np.mean(child_x), -depth * spacing_y)
            else:
                pos[node.id] = (x_counter[0], -depth * spacing_y)
                x_counter[0] += spacing_x

        assign_positions(self.root, 0, pos, x_counter)

        return pos

    def get_all_node_boundaries(self, node):
        """
        Retrieves all node boundaries in the current QuadTree.
        """
        boundaries = []

        def traverse(n):
            if n.is_divided or n.body is not None:
                boundaries.append(n.boundary)
                if n.is_divided:
                    for child in n.children:
                        if child is not None:
                            traverse(child)

        traverse(node)
        return boundaries

## Animation Functions

### Objective
This code is designed to animate the construction of a **QuadTree** by showing its subdivision process as particles are added. It provides visual feedback for how the QuadTree partitions space at different levels of depth, with a focus on the incremental steps of subdivision.

### Key Components

#### 1. `get_subdivision_steps_by_depth(node)`
- **Purpose**: This function traverses the QuadTree, collecting the subdivisions at each depth level and organizing them into lists.
- **Process**:
  - The function performs a breadth-first traversal of the QuadTree.
  - For each node that is subdivided, it collects details like the node's boundary, its ID, and whether it contains a particle.
  - The subdivisions are grouped by depth, ensuring that nodes created at the same level are grouped together.
- **Returns**: A list of lists, where each sublist contains information about the nodes at a specific depth.

#### 2. `animate_quadtree_construction(subdivision_steps, particles_list, R, save=False, filename='quadtree_animation.gif')`
- **Purpose**: This function creates an animation that visually represents the construction of the QuadTree over time.
- **Parameters**:
  - `subdivision_steps`: The output from `get_subdivision_steps_by_depth`, containing details about the nodes created at each level.
  - `particles_list`: A list of the particles (with position) that are present in the QuadTree.
  - `R`: Radius of the spherical region.
  - `save`: A boolean flag indicating whether to save the animation as a GIF.
  - `filename`: Filename to use if the animation is saved.
- **Process**:
  - **Figure Setup**: The function initializes a **matplotlib** figure and sets axis limits to fit the spatial region.
  - **Scatter Plot**: The particles are plotted at their positions, shown as blue points.
  - **Animation Initialization (`init`)**: This clears any existing plot elements (rectangles and texts).
  - **Animation Update (`update`)**: As the animation progresses, rectangles representing the boundaries of the subdivided nodes are drawn, with their corresponding node IDs displayed if they contain a particle.
  - **Animation Creation**: Uses **FuncAnimation** to step through the subdivisions and render the QuadTree construction incrementally.
  - **Saving**: If the `save` flag is `True`, it attempts to save the animation using ImageMagick.

### Animation Process
1. **Initialization**: The function prepares the axes, places particles, and prepares containers for rectangle patches and text annotations.
2. **Update Function**: For each frame of the animation:
   - It plots the boundary rectangles for nodes at the current depth.
   - If a node contains a particle, it displays the node's ID at the center of the rectangle.
3. **Display**: The final animated result is shown, and optionally, the animation can be saved as a GIF.

### Summary
This animation function provides a dynamic way to visualize how a **QuadTree** partitions space as particles are added, showing the hierarchical structure's development step by step. The visual representation helps in understanding how the tree's structure evolves, especially useful in simulations involving spatial partitioning.

In [ ]:
# ===============================
# Animation Functions
# ===============================

def get_subdivision_steps_by_depth(node):
    """
    Traverses the QuadTree and groups subdivision steps by depth.
    
    Returns:
    - List of lists, where each sublist contains tuples of (boundary, node_id, has_particle) for child nodes created at the same depth.
    """
    depth_steps = defaultdict(list)
    queue = [node]
    while queue:
        current = queue.pop(0)
        if current.is_divided:
            # Record the boundaries, IDs, and particle presence of child nodes
            for child in current.children:
                if child is not None:
                    #has_particle = child.body is not None
                    has_particle = child.mass > 0
                    depth_steps[child.depth].append((child.boundary, child.id, has_particle))
                    queue.append(child)
    # Sort the steps by depth
    sorted_depths = sorted(depth_steps.keys())
    subdivision_steps = [depth_steps[depth] for depth in sorted_depths]
    return subdivision_steps

def animate_quadtree_construction(subdivision_steps, particles_list, R, save=False, filename='quadtree_animation.gif'):
    """
    Creates an animation showing the construction of the QuadTree.
    
    Parameters:
    - subdivision_steps: List of subdivision steps (each step is a list of (boundary, node_id, has_particle) tuples)
    - particles_list: List of particle positions (numpy arrays)
    - R: Radius of the disk
    - save: Boolean indicating whether to save the animation
    - filename: Filename for the saved animation
    """
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(-R, R)
    ax.set_ylim(-R, R)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title('QuadTree Construction Animation')
    ax.set_aspect('equal')

    # Plot all particles at the start
    scat = ax.scatter([p[0] for p in particles_list],
                      [p[1] for p in particles_list],
                      color='blue', s=100, label='Particles', zorder=5)
    #ax.legend(loc='upper right')

    # List to hold rectangle patches and text annotations
    rectangles = []
    texts = []

    def init():
        """
        Initializes the animation by clearing any existing rectangles and texts.
        """
        for rect in rectangles:
            rect.remove()
        rectangles.clear()
        for txt in texts:
            txt.remove()
        texts.clear()
        return [scat] + rectangles + texts

    def update(frame):
        """
        Updates the animation by adding the next set of subdivisions.
        """
        if frame < len(subdivision_steps):
            current_depth = frame  # Assuming subdivision_steps are ordered by depth
            color = depth_colors[(current_depth + 1) % len(depth_colors)]
            for boundary, node_id, has_particle in subdivision_steps[frame]:
                x_min, x_max, y_min, y_max = boundary
                rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                     fill=None, edgecolor=color, linewidth=1.5, zorder=1)
                ax.add_patch(rect)
                rectangles.append(rect)
                
                # Only display the node ID if the node contains a particle
                if has_particle:
                    cx = (x_min + x_max) / 2
                    cy = (y_min + y_max) / 2
                    txt = ax.text(cx, cy, str(node_id), color=color, fontsize=20, ha='center', va='center', zorder=2)
                    texts.append(txt)
        return [scat] + rectangles + texts

    ani = FuncAnimation(fig, update, frames=len(subdivision_steps),
                        init_func=init, blit=False, repeat=False, interval=1000)

    if save:
        try:
            ani.save(filename, writer='pillow', fps=2)
            print(f"Animation saved as {filename}")
        except Exception as e:
            print(f"Failed to save animation: {e}")
            print("Ensure that Pillow is installed and added to your system's PATH.")
    
    plt.show()

## Data Generation and QuadTree Construction

### Objective
This code generates random particles within a 2D spherical region and constructs a **QuadTree** using the **Barnes-Hut algorithm**. The QuadTree is used to efficiently partition the space and is visualized through both an animation of the construction process and a final tree structure.

### Key Components

#### 1. **Data Generation and QuadTree Construction**
- **Purpose**: This section initializes the simulation by generating random particles, creating a **QuadTree**, and inserting the particles into the tree.
- **Process**:
  - **Random Particle Generation**:
    - `theta`: Random angles are generated uniformly between \(0\) and \(2\pi\).
    - `r`: Random radial distances are generated uniformly between \(0\) and \(R^2\), followed by a square root transformation to ensure uniform distribution over a circular region.
    - `x_positions` and `y_positions`: These are the Cartesian coordinates of the particles, converted from polar coordinates.
  - **Body Instances**:
    - Each particle is treated as a **Body** object, with equal mass distributed among the total mass (`Mtot`).
  - **QuadTree Construction**:
    - A **Barnes-Hut** tree (`bh_tree`) is created with a specified boundary and maximum depth.
    - All particles are inserted into the tree using the `insert` method.

#### 2. **Subdivision Steps**
- **Purpose**: The subdivision process is recorded and grouped by depth level.
- **Process**:
  - The function `get_subdivision_steps_by_depth` traverses the **QuadTree** and collects details of the subdivisions, such as the boundaries of nodes and whether they contain particles. These subdivisions are organized by the depth at which they occur.

#### 3. **QuadTree Construction Animation**
- **Purpose**: This section animates the construction of the **QuadTree**, showing how space is partitioned as particles are inserted.
- **Process**:
  - The function `animate_quadtree_construction` animates the QuadTree subdivision process using the recorded steps.
  - **Saving the Animation**: If the `save` flag is set to `True`, the animation is saved as a GIF using **ImageMagick**.
  - The animation shows the boundaries of subdivided nodes, with node IDs displayed for nodes that contain particles.

#### 4. **Final QuadTree Visualization (Optional)**
- **Purpose**: This section offers two visualization options to observe the final state of the QuadTree:
  - **Spatial Visualization**:
    - The final state of the **QuadTree**'s spatial division can be plotted using the `visualize_spatial_tree` method. This shows the spatial boundaries of nodes and the positions of particles within the region.
  - **Hierarchical Tree Visualization**:
    - The hierarchical structure of the **QuadTree** is displayed using **networkx**. This allows for a visual representation of the relationships between parent and child nodes, where the nodes are placed hierarchically.

In [ ]:
# ===============================
# Data Generation and QuadTree Construction
# ===============================

# Settings
Mtot = 10            # Total mass
num_bodies = 15     # Number of particles (increased for better visualization)
R = 5                # Radius of the disk

# Generate random positions

theta = np.random.uniform(0, 2 * np.pi, num_bodies)  # Random angles (0~2π)
r = np.sqrt(np.random.uniform(0, R**2, num_bodies))  # Random radii (0~R)

# Convert from polar to Cartesian coordinates
x_positions = r * np.cos(theta)
y_positions = r * np.sin(theta)

# Create Body instances (assuming equal mass for simplicity)
mass = Mtot / num_bodies
bodies = [Body(position=(x, y), mass=mass) for x, y in zip(x_positions, y_positions)]

# Create the Barnes-Hut tree with a maximum depth to prevent infinite subdivisions
boundary = (-R, R, -R, R)
bh_tree = BarnesHut(boundary, max_depth=10)

# Insert all particles into the QuadTree
for body in bodies:
    bh_tree.insert(body)

# Collect subdivision steps grouped by depth
subdivision_steps = get_subdivision_steps_by_depth(bh_tree.root)

# Collect particle positions
particles_list = [body.position for body in bodies]

# ===============================
# Animate the QuadTree Construction
# ===============================

# To display the animation without saving:
# animate_quadtree_construction(subdivision_steps, particles_list, R, save=False)

# To save the animation as GIF (requires Pillow):
animate_quadtree_construction(subdivision_steps, particles_list, R, save=True, filename='quadtree_animation.gif')

# ===============================
# Optional: Visualize the Final QuadTree Structure
# ===============================

# 1. Visualize the final spatial QuadTree division
#plt.figure(figsize=(10, 10))
#ax = plt.gca()
#bh_tree.visualize_spatial_tree(bh_tree.root, ax)  # Display the spatial tree
#plt.scatter([body.position[0] for body in bodies], [body.position[1] for body in bodies], s=100, color='blue', label='Particles', zorder=5)
#plt.xlim(-R, R)
#plt.ylim(-R, R)
#plt.xlabel('X')
#plt.ylabel('Y')
#plt.title('Final Spatial QuadTree Division')
#plt.axis('equal')
##plt.legend()
#plt.show()

# 2. Visualize the hierarchical tree structure
bh_tree.visualize_tree_structure(font_size=20)